In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import re
from explain.eval.tools.bio.localization import SCLArgs, SCLVerifier

/rxrx/data/user/lu.zhu/hooke-explain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-12 19:35:43.101 | INFO     | explain.llm._client:__init__:472 - Initialized OpenAI client with model gpt-4.1
2025-08-12 19:35:43.102 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: openai


### Load explain example

In [2]:
TEST_DATA_PATH = (
    "/mnt/ps/home/CORP/lu.zhu/project/hooke-explain/data/curation_v1/results/structure-explain-results-v3-gemini2-5.csv"
)

data = pd.read_csv(TEST_DATA_PATH)

In [3]:
scl_rows = [expl for row in data["explain"] for expl in row.split("\n") if "localises_to" in expl]
scl_rows

['  localises_to(id="n5", entity="STAT6", from_loc="cytoplasm", to_loc="nucleus", direction="down", via="decreased phosphorylation and dimerization")',
 '  localises_to(id="n6", entity="STAT1/STAT3/STAT6", from_loc="cytoplasm", to_loc="nucleus", direction="down", via="lack of phosphorylation")',
 '  localises_to(id="n5", entity="STAT proteins", from_loc="cytoplasm", to_loc="nucleus", direction="down", via="lack of phosphorylation and dimerization", confidence="high")',
 '  localises_to(id="n6", entity="STAT6", from_loc="cytoplasm", to_loc="nucleus", direction="down", via="reduced phosphorylation and dimerization")',
 '  localises_to(id="n7", entity="STAT6 dimer", from_loc="cytoplasm", to_loc="nucleus", direction="down", via="reduced STAT6 phosphorylation")',
 '  localises_to(id="n4", entity="STAT6", from_loc="cytoplasm", to_loc="nucleus", direction="down", via="prevention of dimerization and nuclear translocation due to reduced phosphorylation", confidence="high")',
 '  localises_to(id

In [4]:
data

,index,question,thinking,answer,explain,dag,raw_response,success,error,input_perturbation,input_report_text
0,0,How does the following perturbation influence ...,The task is to explain the mechanism of action...,"Bevacizumab, a recombinant humanized monoclona...","set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,# Comprehensive Mechanistic Report: Bevacizuma...
1,1,How does the following perturbation influence ...,1. **Context Understanding**: The initial con...,Nintedanib is a multi-tyrosine kinase inhibito...,"set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n0"", ""n1"", relation=""causal"")\nedge(""n1""...",<think>\n1. **Context Understanding**: The in...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Nintedanib's Inhibitor...
2,2,How does the following perturbation influence ...,The user wants a structured explanation of Apa...,Apatinib is a small-molecule tyrosine kinase i...,"set_context(id=""n0"", cell_type=""vascular endot...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe user wants a structured explanati...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Apatinib in VEGF-Media...
3,3,How does the following perturbation influence ...,The task is to explain the mechanism of action...,Lenvatinib is a multi-targeted tyrosine kinase...,"set_context(id=""n0"", cell_type=""vascular endot...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Impact of Lenvatinib o...
4,4,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,"Motesanib, a multi-targeted tyrosine kinase in...","set_context(id=""n0"", cell_type=""endothelial ce...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,# Motesanib: A Multi-Targeted Tyrosine Kinase ...
...,...,...,...,...,...,...,...,...,...,...,...
171,171,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,"Solcitinib, an ATP-competitive inhibitor, bind...","set_context(id=""n0"", disease=""inflammatory dis...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Report: Solcitinib's Impact on...
172,172,How does the following perturbation influence ...,The task is to explain the mechanism of action...,TG 101209 is an ATP-competitive inhibitor that...,"set_context(id=""n0"", cell_type=""immune cells"",...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Analysis of TG 101209 in IFN-g...
173,173,How does the following perturbation influence ...,The task is to explain the mechanism of action...,"TG 101348, an ATP-competitive inhibitor, prima...","set_context(id=""n0"", cell_type=""immune cells"",...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n2""...",<think>\nThe task is to explain the mechanism ...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic Analysis of TG 101348 in IFNγ-...
174,174,How does the following perturbation influence ...,The user wants to understand the mechanism of ...,AZ 960 is a potent and selective ATP-competiti...,"set_context(id=""n0"", cell_type=""various"", dise...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n3""...",<think>\nThe user wants to understand the mech...,True,NaN,{'context': {'perturbation_type': 'soluble fac...,\n# Mechanistic R

In [5]:
scl_rows = [
    (pert, expl)
    for i, (pert, row) in data[["input_perturbation", "explain"]].iterrows()
    for expl in row.split("\n")
    if "localises_to" in expl
]

# extract the scl info from explain
scl_data = []
for pert, expl in scl_rows:
    entity = re.search(r'entity\s*=\s*["\']([^"\']+)["\']', expl).group(1)
    from_loc = re.search(r'from_loc\s*=\s*["\']([^"\']+)["\']', expl)
    if from_loc:
        from_loc = from_loc.group(1).strip()
    to_loc = re.search(r'to_loc\s*=\s*["\']([^"\']+)["\']', expl).group(1)
    mechanism = re.search(r'mechanism\s*=\s*["\']([^"\']*)["\']', expl)
    if mechanism:
        mechanism = mechanism.group(1).strip()
    modification = re.search(r'via\s*=\s*["\']([^"\']+)["\']', expl)
    if modification:
        modification = modification.group(1).strip()
    scl_data.append([pert, entity, from_loc, to_loc, mechanism, modification])

scl_data = pd.DataFrame(scl_data, columns=["perturbation", "entity", "from_loc", "to_loc", "mechanism", "modification"])

In [6]:
scl_data = scl_data.drop_duplicates(subset=["entity", "from_loc", "to_loc"])
scl_data

,perturbation,entity,from_loc,to_loc,mechanism,modification
0,{'context': {'perturbation_type': 'soluble fac...,STAT6,cytoplasm,nucleus,None,decreased phosphorylation and dimerization
1,{'context': {'perturbation_type': 'soluble fac...,STAT1/STAT3/STAT6,cytoplasm,nucleus,None,lack of phosphorylation
2,{'context': {'perturbation_type': 'soluble fac...,STAT proteins,cytoplasm,nucleus,None,lack of phosphorylation and dimerization
4,{'context': {'perturbation_type': 'soluble fac...,STAT6 dimer,cytoplasm,nucleus,None,reduced STAT6 phosphorylation
10,{'context': {'perturbation_type': 'soluble fac...,STAT1/3/6,cytoplasm,nucleus,None,impaired dimerization due to lack of phosphory...
14,{'context': {'perturbation_type': 'soluble fac...,STAT dimers,cytoplasm,nucleus,None,prevention of phosphorylation and dimerization
16,{'context': {'perturbation_type': 'loss-of-fun...,phosphorylated Smad2/3,cytoplasm,nucleus,None,phosphorylation-induced nuclear translocation
17,{'context': {'perturbation_type': 'loss-of-fun...,Smad2/3,nucleus,cytoplasm,None,reduced phosphorylation preventing nuclear tra...
18,{'context': {'perturbation_type': 'loss-of-fun...,SMAD2/3 complex,cytoplasm,nucleus,None,reduced phosphorylation
19,{'context': {'perturbation_type': 'loss-of-fun...,phosphorylated SMAD2/3,cytoplasm,nucleus,None,reduced phosphorylation


### Define the SCLArgs

In [7]:
row = scl_data.iloc[0]
row

perturbation    {'context': {'perturbation_type': 'soluble fac...
entity                                                      STAT6
from_loc                                                cytoplasm
to_loc                                                    nucleus
mechanism                                                    None
modification           decreased phosphorylation and dimerization
Name: 0, dtype: object

In [8]:
sclv = SCLVerifier()

In [ ]:
for _, row in scl_data.iterrows():
    print("row: ", row.to_dict())
    args = SCLArgs(
        protein_entity=row["entity"],
        from_loc=row["from_loc"],
        to_loc=row["to_loc"],
        mechanism=row["mechanism"],
        modification=row["modification"],
    )
    # run verifier
    print(sclv._tool_logic(args))
    
    break

row:  {'perturbation': "{'context': {'perturbation_type': 'soluble factor', 'description': 'Soluble factor addition of IL-13', 'cell_type': 'N/A', 'disease_model': 'Asthma and allergy - IL-13'}, 'perturbation': {'type': 'chemical', 'smiles': None, 'name': 'Lebrikizumab', 'target': 'IL13', 'moa_type': 'antibody'}}", 'entity': 'STAT6', 'from_loc': 'cytoplasm', 'to_loc': 'nucleus', 'mechanism': None, 'modification': 'decreased phosphorylation and dimerization'}
2025-08-12 19:35:48,168 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine


2025-08-12 19:35:48.193 | INFO     | explain.eval.tools.bio.entity:retrieve_identifiers:152 - NER model info: hunflair2
2025-08-12 19:36:10.236 | INFO     | explain.eval.tools.bio.entity:_ID_mapping:102 - ID mapping.


(0.3, {'protein_entity': 'STAT6', 'from_loc': 'cytoplasm', 'to_loc': 'nucleus', 'mechanism': None, 'modification': 'decreased phosphorylation and dimerization', 'verification_status': 'NOT_VERIFIED'})
